# RG3 가스 15모델 비교
동일 데이터와 피처, 학습 내부 판정 기준. 모델 선택 편향 주의.

In [1]:
from pathlib import Path
import json,warnings
import numpy as np,pandas as pd
from scipy.special import expit
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier,HistGradientBoostingClassifier,GradientBoostingClassifier,AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression,RidgeClassifier,SGDClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis,QuadraticDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score,confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
nb=json.loads((root/'notebooks/22_rg3_nested_signal_ensemble.ipynb').read_text(encoding='utf-8'))
exec(''.join(nb['cells'][1]['source']).split('scores=[];thresholds=[];foldlogs=[]')[0])
models={
'RandomForest':RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=3,class_weight='balanced_subsample',random_state=42,n_jobs=-1),
'ExtraTrees':ExtraTreesClassifier(n_estimators=200,max_depth=8,min_samples_leaf=3,class_weight='balanced',random_state=42,n_jobs=-1),
'HistGradientBoosting':HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=7,min_samples_leaf=10,l2_regularization=1,class_weight='balanced',random_state=42),
'GradientBoosting':GradientBoostingClassifier(n_estimators=100,max_depth=2,min_samples_leaf=5,random_state=42),
'AdaBoost':AdaBoostClassifier(n_estimators=100,learning_rate=.5,random_state=42),
'DecisionTree':DecisionTreeClassifier(max_depth=4,min_samples_leaf=8,class_weight='balanced',random_state=42),
'LogisticRegression':LogisticRegression(C=1,class_weight='balanced',max_iter=2000,random_state=42),
'RidgeClassifier':RidgeClassifier(alpha=10,class_weight='balanced'),
'SGDLogistic':SGDClassifier(loss='log_loss',alpha=.01,class_weight='balanced',max_iter=3000,tol=1e-4,random_state=42),
'SVC_RBF':SVC(C=1,kernel='rbf',class_weight='balanced'),
'SVC_Linear':SVC(C=1,kernel='linear',class_weight='balanced'),
'KNN':KNeighborsClassifier(n_neighbors=15,weights='distance'),
'GaussianNB':GaussianNB(var_smoothing=1e-6),
'LDA':LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto'),
'QDA':QuadraticDiscriminantAnalysis(reg_param=.2)}
trainrescache={};mpcache={};warnrows=[]
def train_res(train):
    key=tuple(train)
    if key in trainrescache:return trainrescache[key]
    out=np.full((len(train),4),np.nan)
    for b in groups.iloc[train].unique():
        mask=groups.iloc[train].eq(b).to_numpy()
        ref=train[~mask];query=train[mask]
        if len(ref) and (y[ref]==0).any():out[mask]=res_features(ref,query)
    trainrescache[key]=out
    return out
def fitted_score(name,x,yy,q):
    if len(np.unique(yy))<2:return np.full(len(q),float(yy[0]) if len(yy) else 0.)
    steps=[SimpleImputer(strategy='median',keep_empty_features=True)]
    if name not in ['RandomForest','ExtraTrees','HistGradientBoosting','GradientBoosting','AdaBoost','DecisionTree']:steps.append(StandardScaler())
    pipe=make_pipeline(*steps,clone(models[name]))
    with warnings.catch_warnings(record=True) as ww:
        warnings.simplefilter('always')
        pipe.fit(x,yy)
        if name=='GaussianNB':pipe.steps[-1][1].var_=np.maximum(pipe.steps[-1][1].var_,1e-8)
        if hasattr(pipe,'predict_proba'):score=pipe.predict_proba(q)[:,1]
        else:score=expit(pipe.decision_function(q))
        for w in ww:warnrows.append(dict(model=name,category=w.category.__name__,message=str(w.message)))
    assert np.isfinite(score).all()
    return score
def model_pair(name,train,test):
    key=(name,tuple(train),tuple(test))
    if key in mpcache:return mpcache[key]
    assert set(groups.iloc[train]).isdisjoint(groups.iloc[test])
    a=fitted_score(name,d.iloc[train][stdcols],y[train],d.iloc[test][stdcols])
    b=fitted_score(name,train_res(train),y[train],res_features(train,test))
    mpcache[key]=np.column_stack([a,b])
    return mpcache[key]
scores=[];logs=[];errors=[]
for name in models:
    try:
        local=[];locallogs=[]
        for scheme in ['leave_one_block_out','past_blocks_only']:
            for j,block in enumerate(blocks):
                test=np.flatnonzero(groups.eq(block))
                train=np.flatnonzero(~groups.eq(block)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
                if len(np.unique(y[train]))<2:continue
                outer=model_pair(name,train,test)
                inner=np.full((len(train),2),np.nan)
                for ib in groups.iloc[train].unique():
                    mask=groups.iloc[train].eq(ib).to_numpy()
                    if (~mask).any():inner[mask]=model_pair(name,train[~mask],train[mask])
                locallogs.append(dict(model=name,scheme=scheme,block=block,train=len(train),train_gas=int(y[train].sum()),test=len(test),test_gas=int(y[test].sum())))
                for channel,k in [('std4',0),('residual4',1),('max_union',None)]:
                    predscore=outer[:,k] if k is not None else outer.max(axis=1)
                    cal=inner[:,k] if k is not None else inner.max(axis=1)
                    normal=cal[(y[train]==0)&np.isfinite(cal)]
                    for budget in [.01,.03,.05]:
                        cut=float(np.quantile(normal,1-budget,method='higher')) if len(normal) else 1.
                        for ii,z in enumerate(test):local.append(dict(model=name,scheme=scheme,channel=channel,budget=budget,TimeStamp=d.TimeStamp.iloc[z],block=block,actual=int(y[z]),score=float(predscore[ii]),threshold=cut,prediction=int(predscore[ii]>cut)))
        scores.extend(local);logs.extend(locallogs)
        print('Completed',name,flush=True)
    except Exception as e:
        errors.append(dict(model=name,error=repr(e)));print('FAILED',name,repr(e),flush=True)
pp=pd.DataFrame(scores);rows=[]
for (scheme,name,channel,budget),g in pp.groupby(['scheme','model','channel','budget'],sort=False):
    tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
    rows.append(dict(scheme=scheme,model=name,channel=channel,budget=budget,shots=len(g),gas=int(g.actual.sum()),AP=average_precision_score(g.actual,g.score),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),precision=tp/(tp+fp) if tp+fp else 0,recall=tp/(tp+fn),FPR=fp/(fp+tn)))
metrics=pd.DataFrame(rows)
pp.to_csv(tables/'rg3_15models_predictions.csv',index=False,encoding='utf-8-sig')
metrics.to_csv(tables/'rg3_15models_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(logs).to_csv(tables/'rg3_15models_fold_audit.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(warnrows,columns=['model','category','message']).to_csv(tables/'rg3_15models_warnings.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(errors,columns=['model','error']).to_csv(tables/'rg3_15models_errors.csv',index=False,encoding='utf-8-sig')
# Verify exact old RF union before ranking any new models.
old=pd.read_csv(tables/'rg3_nested_ensemble_predictions.csv',parse_dates=['TimeStamp'])
chk=pp.loc[pp.model.eq('RandomForest')&pp.channel.eq('max_union')&pp.budget.eq(.01)].merge(old.loc[old.model.eq('max_union'),['scheme','TimeStamp','score']],on=['scheme','TimeStamp'],validate='one_to_one',suffixes=('_new','_old'))
assert len(chk)==1023 and np.allclose(chk.score_new,chk.score_old)
summary=metrics.loc[metrics.channel.eq('max_union')&metrics.budget.eq(.01)].sort_values(['scheme','AP'],ascending=[True,False])
lines=['# RG3 가스: 15가지 모델 비교','',
'RH 정상566/가스17. 기존 흔들림4개와 관계잔차4개는 그대로 유지하고, 각 모델마다 두 분류기를 따로 학습한다. 두 점수 중 최대값 결합과 두 단독 채널을 모두 저장한다. 미성형은 제외한다.',
'잔차 학습은 학습 구간 내 정상 참조를 이용하며, 분류기는 정상/가스 라벨을 모두 사용한다. 구간 제외 검증과 과거 구간만 학습한 검증을 별도 보고한다. imputer/scaler는 학습 데이터에만 적합한다.',
'판정 기준은 학습 내부 구간검증 정상 점수의 상위1/3/5% 경계로 고정하며, 외부 평가 정상의 오탐률을 보장하지 않는다. SVC/Ridge의 결정점수는 sigmoid로 변환하지만 확률 보정이 아니다. 모델별 출력 척도가 달라 max 결합의 효과도 함께 달라질 수 있다.',
'하이퍼파라미터는 이번 비교 전에 고정했으며 15모델 중 최고를 동일 평가자료에서 선택하므로 독립적인 최종 평가가 아니다. 기존 반복탐색 영향도 있다. AP는 PR-AUC와 관련되지만 동일한 계산 방식은 아니다.',
'XGBoost/LightGBM/CatBoost는 설치되어 있지 않아 제외하고 scikit-learn 15모델을 비교한다. 새로운 패키지는 설치하지 않았다.',
'이전10샷 원본 센서 이력을 쓸 수 있다는 조건이며 대회 테스트 이력 제공 여부는 확인 필요. RPM 정의 불확실한 기존 잔차는 원인 설명에 쓰지 않는다. 초반 두구간만 내부 학습에 남으면 잔차 학습특성이 결측일 수 있으며 학습집합 median/0 대체를 한다.',
'','## 최대점수 결합, 보수적 기준','',
'| 평가 | 모델 | AP | TP | FP | FN | TN |','|---|---|---:|---:|---:|---:|---:|']
for r in summary.itertuples():lines.append(f'| {r.scheme} | {r.model} | {r.AP:.4f} | {r.TP} | {r.FP} | {r.FN} | {r.TN} |')
lines+=['','전체 채널 및 기준별 결과는 rg3_15models_metrics.csv, 샷별 결과는 rg3_15models_predictions.csv에 저장했다. RF 최대점수 예측1023개는 기존 결과와 정확히 일치하는지 확인했다.',
'',f'실패 모델: {errors}',f'모델 경고 건수: {len(warnrows)}. 경고 파일을 함께 확인한다.']
(root/'reports/rg3_15models_comparison.md').write_text('\n'.join(lines),encoding='utf-8')
print(summary.to_string(index=False))
print('TOP SINGLE CHANNELS')
print(metrics.loc[metrics.budget.eq(.01)].sort_values('AP',ascending=False).head(12).to_string(index=False))


             scheme                model   channel  budget  shots  gas       AP  TP  FP  FN  TN  precision   recall      FPR
leave_one_block_out           ExtraTrees max_union    0.01    583   17 0.174938   2  10  15 556   0.166667 0.117647 0.017668
leave_one_block_out         RandomForest max_union    0.01    583   17 0.168625   4   6  13 560   0.400000 0.235294 0.010601
leave_one_block_out           GaussianNB max_union    0.01    583   17 0.123625   0   1  17 565   0.000000 0.000000 0.001767
leave_one_block_out                  KNN max_union    0.01    583   17 0.119586   2  17  15 549   0.105263 0.117647 0.030035
leave_one_block_out HistGradientBoosting max_union    0.01    583   17 0.080364   2  14  15 552   0.125000 0.117647 0.024735
leave_one_block_out                  QDA max_union    0.01    583   17 0.078589   2  19  15 547   0.095238 0.117647 0.033569
leave_one_block_out              SVC_RBF max_union    0.01    583   17 0.062823   0   8  17 558   0.000000 0.000000 0.014134
